# Fault detection on held-out machine sounds

Recalculate the reported results from saved per-clip scores. This notebook needs NumPy and scikit-learn and runs on a CPU. All thresholds come from healthy training recordings.

## Inspect the evidence

Each fit has an initialization seed and two arrays of scores. All fits share the same ordered clip metadata.

In [1]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/mimii").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
sys.path.insert(0, str(root / "src"))
import json
import numpy as np
from mimii.reproduce_results import results_table, print_table

fits = json.loads(Path("results/fits.json").read_text())
with np.load("results/clip_scores.npz", allow_pickle=False) as evidence:
    print(f"{len(fits)} model/seed combinations")
    for split in ("train", "test"):
        labels = evidence[split + "_label"]
        print(f"{split}: {len(labels):,} clips, {labels.sum():,} abnormal")

14 model/seed combinations
train: 30,948 clips, 0 abnormal
test: 18,714 clips, 9,900 abnormal


## Recompute the comparison

Each machine ID gets a threshold at the 95th percentile of its healthy training scores. AUC and standardized pAUC average across 48 type/ID/SNR groups. Recall and false-alarm rates count recordings. All entries are percentages. See the [metric implementation](code/evaluation/reproduce_results.ipynb).

In [2]:
table = results_table()
print_table(table)

Model                  Seeds      AUC     pAUC   Recall      FPR
MobileNetV3-Small          3    82.68    74.89    62.66    10.96
ResNet-18                  1    90.47    84.21    69.98     6.27
EfficientNet-B0            1    89.68    84.33    69.38     6.63
Compact CNN                3    88.40    81.43    64.51     6.75
Compact mean-only          1    89.23    81.84    65.01     6.57
Wide CNN                   3    87.94    82.06    63.67     6.31
CNN ensemble               1    93.62    88.79    76.69     8.17
XGBoost                    1    91.12    82.45    72.37     7.73


## Inspect initialization variability

Population standard deviations describe the recorded training seeds on the same test set.

In [3]:
for model, row in table.items():
    if row["seeds"] > 1:
        print(f"{model:20s} AUC {row['mean'][0]:.2f} ± {row['sd'][0]:.2f}%")

MobileNetV3-Small    AUC 82.68 ± 0.15%
Compact CNN          AUC 88.40 ± 0.89%
Wide CNN             AUC 87.94 ± 1.36%
